In [4]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch import nn, optim
from tqdm import tqdm

In [3]:
import sys
sys.path.append("/kaggle/input/brats-2d-npy-slices")
from unet_model import UNet

In [5]:
# ✅ Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [11]:
# 📁 Correct base path
base_path = "/kaggle/input/brats-2d-npy-slices/data/data/2d"

# 🖼️ Image and mask directories
train_img_dir = os.path.join(base_path, "train/images")
train_mask_dir = os.path.join(base_path, "train/masks")
val_img_dir = os.path.join(base_path, "val/images")
val_mask_dir = os.path.join(base_path, "val/masks")


In [29]:
# 📦 Import dataset class
from dataset import BrainTumor2DSliceDataset
from torch.utils.data import DataLoader

# 🧠 Create datasets
train_dataset = BrainTumor2DSliceDataset(train_img_dir, train_mask_dir)
val_dataset = BrainTumor2DSliceDataset(val_img_dir, val_mask_dir)

# 🚚 Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)


In [32]:
# dataset.py
import os
import numpy as np
import torch
from torch.utils.data import Dataset

class BrainTumor2DSliceDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_files = sorted(os.listdir(image_dir))
        self.mask_files = sorted(os.listdir(mask_dir))
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image = np.load(os.path.join(self.image_dir, self.image_files[idx])).astype(np.float32)  # (128,128,4)
        mask = np.load(os.path.join(self.mask_dir, self.mask_files[idx])).astype(np.int64)       # (128,128,4) or (128,128)
        
        # If the mask is one-hot encoded (i.e., (128,128,4)), convert it to class indices (i.e., (128,128))
        if mask.ndim == 3 and mask.shape[-1] == 4:  # Check if it's a one-hot encoded mask
            mask = np.argmax(mask, axis=-1).astype(np.int64)  # Convert to class indices (0-3)

        # Convert to torch tensors
        image = torch.from_numpy(image).permute(2, 0, 1)  # (4, 128, 128)
        mask = torch.from_numpy(mask)                     # (128, 128)

        if self.transform:
            image = self.transform(image)

        return image, mask

In [38]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=4):
        super(UNet, self).__init__()

        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.up_conv3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.up_conv2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.up_conv1 = DoubleConv(128, 64)

        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool1(d1))
        d3 = self.down3(self.pool2(d2))

        bn = self.bottleneck(self.pool3(d3))

        up3 = self.up3(bn)
        up3 = self.up_conv3(torch.cat([up3, d3], dim=1))

        up2 = self.up2(up3)
        up2 = self.up_conv2(torch.cat([up2, d2], dim=1))

        up1 = self.up1(up2)
        up1 = self.up_conv1(torch.cat([up1, d1], dim=1))

        return self.final_conv(up1)

In [39]:
model = UNet(in_channels=4, out_channels=4).to(device)
for name, param in model.named_parameters():
    print(name, param.shape)

down1.double_conv.0.weight torch.Size([64, 4, 3, 3])
down1.double_conv.0.bias torch.Size([64])
down1.double_conv.2.weight torch.Size([64, 64, 3, 3])
down1.double_conv.2.bias torch.Size([64])
down2.double_conv.0.weight torch.Size([128, 64, 3, 3])
down2.double_conv.0.bias torch.Size([128])
down2.double_conv.2.weight torch.Size([128, 128, 3, 3])
down2.double_conv.2.bias torch.Size([128])
down3.double_conv.0.weight torch.Size([256, 128, 3, 3])
down3.double_conv.0.bias torch.Size([256])
down3.double_conv.2.weight torch.Size([256, 256, 3, 3])
down3.double_conv.2.bias torch.Size([256])
bottleneck.double_conv.0.weight torch.Size([512, 256, 3, 3])
bottleneck.double_conv.0.bias torch.Size([512])
bottleneck.double_conv.2.weight torch.Size([512, 512, 3, 3])
bottleneck.double_conv.2.bias torch.Size([512])
up3.weight torch.Size([512, 256, 2, 2])
up3.bias torch.Size([256])
up_conv3.double_conv.0.weight torch.Size([256, 512, 3, 3])
up_conv3.double_conv.0.bias torch.Size([256])
up_conv3.double_conv.2.w

In [43]:
import torch
import torch.optim as optim
import torch.nn as nn

# Model, Loss, Optimizer
model = UNet(in_channels=4, out_channels=4).to(device)  # Ensure in_channels and out_channels match
criterion = nn.CrossEntropyLoss()  # Loss function for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=1e-4)  # Optimizer for training

num_epochs = 10

for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0

    for i, (images, masks) in enumerate(train_loader):  # Assuming train_loader is correctly set up
        images, masks = images.to(device), masks.to(device)

        # Add the dummy channel to make it [batch_size, 4, 128, 128]
        images = torch.cat([images, torch.zeros(images.size(0), 1, images.size(2), images.size(3), device=images.device)], dim=1)


        # If masks are one-hot encoded, convert them to class indices
        if masks.dim() == 4:  # Check if masks are one-hot encoded
            masks = torch.argmax(masks, dim=-1)  # Convert to class indices shape [batch_size, height, width]

        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)  # Model outputs raw logits

        # Calculate loss
        loss = criterion(outputs, masks)  # CrossEntropyLoss expects raw logits and class indices

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Print/log progress every 10 steps
        if (i + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

    # Print the average loss for the epoch
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")


Epoch [1/10], Step [10/720], Loss: 1.3170
Epoch [1/10], Step [20/720], Loss: 1.0173
Epoch [1/10], Step [30/720], Loss: 0.4321
Epoch [1/10], Step [40/720], Loss: 0.3696
Epoch [1/10], Step [50/720], Loss: 0.4669
Epoch [1/10], Step [60/720], Loss: 0.2108
Epoch [1/10], Step [70/720], Loss: 0.1948
Epoch [1/10], Step [80/720], Loss: 0.2153
Epoch [1/10], Step [90/720], Loss: 0.1651
Epoch [1/10], Step [100/720], Loss: 0.1335
Epoch [1/10], Step [110/720], Loss: 0.0635
Epoch [1/10], Step [120/720], Loss: 0.1366
Epoch [1/10], Step [130/720], Loss: 0.0783
Epoch [1/10], Step [140/720], Loss: 0.1024
Epoch [1/10], Step [150/720], Loss: 0.1616
Epoch [1/10], Step [160/720], Loss: 0.0514
Epoch [1/10], Step [170/720], Loss: 0.1297
Epoch [1/10], Step [180/720], Loss: 0.0834
Epoch [1/10], Step [190/720], Loss: 0.0935
Epoch [1/10], Step [200/720], Loss: 0.0955
Epoch [1/10], Step [210/720], Loss: 0.1307
Epoch [1/10], Step [220/720], Loss: 0.1471
Epoch [1/10], Step [230/720], Loss: 0.0963
Epoch [1/10], Step [

In [44]:
model.eval()
val_loss = 0.0
with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)

        # Add dummy channel
        dummy_channel = torch.zeros(images.size(0), 1, images.size(2), images.size(3)).to(device)
        images = torch.cat([images, dummy_channel], dim=1)

        if masks.dim() == 4:
            masks = torch.argmax(masks, dim=-1)

        outputs = model(images)
        loss = criterion(outputs, masks)
        val_loss += loss.item()

print(f"Validation Loss: {val_loss / len(val_loader):.4f}")

Validation Loss: 0.0443
